In [2]:
#Amy Independent Research, Fall 2024
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob

import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"

For each model (Model 1 to Model 8), these are the regressors.
Dependent Variable: Average Daily Trip Counts (avgdtc)
1) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Intercept
2) Independent Variables:  Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Intercept
3) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Intercept
4) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Intercept
5) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Membership Proportion * Bike Lane Length (memberp_bl), Intercept
6) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Bike Lane Length (bl), Active Stations (activesta), Membership Proportion (memberp), Electric Bike Proportion (elecbikep), Electric Bike Proportion * Temperature (elecbikep_avgt), Membership Proportion * Bike Lane Length (memberp_bl), Membership Proportion * Electric Bike Proportion (memberp_elecbikep), Intercept
7) Independent Variables: Precipitation (avgpp), Snow Depth (avgsd), Temperature (avgt), Wind Speed (avgws), Active Stations (activesta), Gravity Attractiveness, Intercept

avgdtc: average daily trip counts; avgpp: average precipitation (inch); avgsd: average snow depth (in); avgt: average temperature (F); avgws: average wind speed (mph); bl: total bike lane length (mile); activesta – average number of daily active stations; elecbikep - electric bike proportion; elecbikep_avgt - interaction term between electric bike proportion and average temperature (F); memberp - Membership Proportion; memberp_bl - interaction term between membership proportion and bike lane length (mile); memberp_elecbikep - interaction term between membership proportion and electric bike proportion

AR² refers to the autoregressive process where the current value of the dependent variable (Average Daily Trip Counts) depends on its lagged values up to the second order (2 previous time steps). It captures the effect of temporal autocorrelation in the dependent variable. For example, high trip counts on one day might be followed by high trip counts on the next day (due to momentum or recurring patterns).

ARCH³ refers to the third-order autoregressive conditional heteroskedasticity component of the model. ARCH³ models the conditional variance of the residuals (𝜎t squared) as a function of the squared residuals from up to 3 previous time steps. _cons (Intercept) represents the constant term in the model. It accounts for the baseline level of average daily trip counts when all other variables (independent variables and lagged terms) are zero.

The log-likelihood measures how well the model fits the data. Higher values indicate a better fit.

The Likelihood Ratio Test (LRT) compares the goodness-of-fit between two nested models (e.g., one model is a simplified version of the other). Tests whether adding parameters (e.g., additional lag terms or ARCH terms) significantly improves the model. Adding more lags or ARCH terms should increase the log-likelihood if they improve model fit. A significant p-value for the LRT indicates that the more complex model (with additional parameters) provides a significantly better fit.

Model 1: Observation on average over seven consecutive days in New York City. (ARCH)

Model 2: Observation on average over seven consecutive days in Non-Manhattan.

Model 3: Observation on average over seven consecutive days in Manhattan.

Model 4: Only average weekdays are included in each observation in Manhattan. 

Model 5: Only average weekends are included in each observation in Manhattan.

Model 6: Observation on average over seven consecutive days in Brooklyn.

Model 7: Only average weekdays are included in each observation in Brooklyn. 

Model 8: Only average weekends are included in each observation in Brooklyn.

# Clean Raw Bike Trip Data from CitiBike

In [3]:
file_path = "/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/4193616194.py:4: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [4]:
# Define bounding boxes for each borough
borough_bounds = {
    'Manhattan': {'lat_min': 40.70, 'lat_max': 40.88, 'lng_min': -74.02, 'lng_max': -73.90},
    'Brooklyn': {'lat_min': 40.57, 'lat_max': 40.73, 'lng_min': -74.04, 'lng_max': -73.85},
    'Queens': {'lat_min': 40.54, 'lat_max': 40.80, 'lng_min': -73.95, 'lng_max': -73.70},
    'Bronx': {'lat_min': 40.79, 'lat_max': 40.91, 'lng_min': -73.93, 'lng_max': -73.80},
    'Staten Island': {'lat_min': 40.49, 'lat_max': 40.65, 'lng_min': -74.25, 'lng_max': -74.05}
}

def get_borough(lat, lng):
    for borough, bounds in borough_bounds.items():
        if bounds['lat_min'] <= lat <= bounds['lat_max'] and bounds['lng_min'] <= lng <= bounds['lng_max']:
            return borough
    return 'Other'

# Apply the get_borough function to determine the borough based on latitude and longitude
df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)

In [5]:
# Convert 'started_at' to datetime if not already done
df['started_at'] = pd.to_datetime(df['started_at'])

# Extract year, month, week number, and day of week
df['year'] = df['started_at'].dt.year
df['month'] = df['started_at'].dt.month
df['week_number'] = df['started_at'].dt.isocalendar().week
df['day_of_week'] = df['started_at'].dt.weekday  # 0=Monday, 6=Sunday

# Calculate trip duration in seconds
df['trip_duration'] = (pd.to_datetime(df['ended_at']) - df['started_at']).dt.total_seconds()

/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/2022280727.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['started_at'] = pd.to_datetime(df['started_at'])
/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_63486/2022280727.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['trip_duration'] = (pd.to_datetime(df['ended_at']) - df['started_at']).dt.total_seconds()


In [10]:
# Aggregate weekly data
weekly_aggregated = df.groupby(['year', 'week_number', 'month', 'borough']).agg(
    total_trips=('ride_id', 'count'),  # Total trips
    weekday_trips=('ride_id', lambda x: x[df.loc[x.index, 'day_of_week'] < 5].count()),  # Weekdays (Mon-Fri)
    weekend_trips=('ride_id', lambda x: x[df.loc[x.index, 'day_of_week'] >= 5].count()),  # Weekends (Sat-Sun)
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),  # Electric bike trips
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),  # Classic bike trips
    member_rides=('member_casual', lambda x: (x == 'member').sum()),  # Member trips
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),  # Casual trips
    avg_trip_duration=('trip_duration', 'mean'),  # Average trip duration
    unique_start_stations=('start_station_id', pd.Series.nunique),  # Unique start stations
    unique_end_stations=('end_station_id', pd.Series.nunique)  # Unique end stations
).reset_index()

In [11]:
# Calculate proportions and averages
weekly_aggregated['electric_bike_proportion'] = weekly_aggregated['electric_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['classic_bike_proportion'] = weekly_aggregated['classic_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['member_proportion'] = weekly_aggregated['member_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['casual_proportion'] = weekly_aggregated['casual_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['avg_daily_trips'] = weekly_aggregated['total_trips'] / 7
weekly_aggregated['avg_weekday_trips'] = weekly_aggregated['weekday_trips'] / 5  # Average weekday trips
weekly_aggregated['avg_weekend_trips'] = weekly_aggregated['weekend_trips'] / 2  # Average weekend trips

In [13]:
# Add a week label (optional, if needed for visualization)
weekly_aggregated['week_label'] = weekly_aggregated.apply(
    lambda row: f"Week {row['week_number']} - {row['month']:02d} - {row['year']}", axis=1
)

# Filter data to include only years 2021 and later
weekly_aggregated = weekly_aggregated[weekly_aggregated['year'] >= 2021]

# Reset the index for the final DataFrame
weekly_aggregated.reset_index(drop=True, inplace=True)

# Display the aggregated DataFrame
print(weekly_aggregated)

    year  week_number  month    borough  total_trips  weekday_trips  \
0   2021            1      1      Bronx          413            324   
1   2021            1      1   Brooklyn        29635          20775   
2   2021            1      1  Manhattan       205760         150601   
3   2021            2      1      Bronx          484            346   
4   2021            2      1   Brooklyn        33540          23335   
5   2021            2      1  Manhattan       229788         166959   
6   2021            3      1      Bronx          321            246   
7   2021            3      1   Brooklyn        30743          23201   
8   2021            3      1  Manhattan       209942         163205   
9   2021            4      1      Bronx          234            161   
10  2021            4      1   Brooklyn        23424          17445   
11  2021            4      1  Manhattan       167336         128696   
12  2021           53      1      Bronx          164             56   
13  20